# Задание 5

## Часть 1 — Анализ FastQC

---

1. С помощью `SRA Explorer` и slurm скрипта скачал 4 файла (`ERR14230587.fastq.gz`,  `ERR14230588.fastq.gz`,  `ERR14230598.fastq.gz`,  `ERR14230605.fastq.gz`) из `PRJEB84057` в `папку hw5/data/raw`:

**download_data.slurm**
```bash 
#!/bin/bash

#SBATCH --job-name=download
#SBATCH --cpus-per-task=8
#SBATCH --mem=1gb
#SBATCH --time=11:00:00
#SBATCH --output=download_%j.out
#SBATCH --error=download_%j.err
#SBATCH --partition=AMD9554

mkdir -p ../data/raw
cd ../data/raw

curl -L ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR142/088/ERR14230588/ERR14230588.fastq.gz -o ERR14230588.fastq.gz
curl -L ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR142/005/ERR14230605/ERR14230605.fastq.gz -o ERR14230605.fastq.gz
curl -L ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR142/087/ERR14230587/ERR14230587.fastq.gz -o ERR14230587.fastq.gz
curl -L ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR142/098/ERR14230598/ERR14230598.fastq.gz -o ERR14230598.fastq.gz
```

2. Используя знания про SLURM, написал скрипт, запускающий fastqc на скачанных данных и агрегирует все отчеты в один с помощью multiqc

**fastqc_raw.slurm**
```bash 
#!/bin/bash

#SBATCH --job-name=fastqc_raw
#SBATCH --output=fastqc_raw.out
#SBATCH --error=fastqc_raw.err
#SBATCH --cpus-per-task=8
#SBATCH --mem=8G
#SBATCH --time=02:00:00
#SBATCH --partition=AMD9554

source ~/.bashrc
conda activate bioinf

mkdir -p ~/hw/hw5/results/fastqc_raw
mkdir -p ~/hw/hw5/results/multiqc_raw

fastqc ~/hw/hw5/data/raw/*.fastq.gz \
       -o ~/hw/hw5/results/fastqc_raw \
       -t 8

multiqc ~/hw/hw5/results/fastqc_raw \
       -o ~/hw/hw5/results/multiqc_raw
```

3. Выгрузил `multiqc_report.html` в директорию `hw5` и попытался проанализировать полученные данные.

```bash
scp studfbmf02_06@calc.cod.phystech.edu:/home/STUDY/FBMF/studfbmf02_06/hw/hw5/results/multiqc_raw/multiqc_report.html .
```

**Вывод:** 

Анализ файлов с использованием fastqc и multiqc показал, что общее качество данных находится на хорошем уровне, однако присутствуют некоторые проблемы

Распределение качества ридов (Sequence Quality Histograms и Per Sequence Quality Scores) демонстрирует высокие значения Phred score ($\approx$ 37), что свидетельствует о хорошем качестве секвенирования. GC-состав образцов стабилен ( $\sim$ 45-46%), что соответствует ожидаемым значениям, однако графики Per Sequence GC Content в ряде случаев отклоняются от нормального распределения, что может указывать на возможную слабую контаминацию

Анализ Per Base Sequence Content показывает наличие отклонений (1 warn и 3 fail) во всех образцах, что характерно для смещения нуклеотидного состава в начале ридов и может быть связано с особенностями протокола секвенирования

Уровень дупликаций варьирует от очень низкого (0.1%) до умеренного ( $\sim$ 20-24%), что находится в допустимых пределах для NGS-данных

Подводя итог, можно сказать, что несмотря на хорошее общее качество ридов, наличие отклонений в распределении нуклеотидов и GC-содержании указывает на необходимость тримминга для улучшения качества данных


## Часть 2 — Тримминг 

---

1. Используя отчеты MultiQC из первой части, определил:

**Вопрос:** Есть ли в ваших данных адаптеры?

**Ответ:** В отчете multqc в конце:
```bash
No samples found with any adapter contamination > 0.1%
```

Это значит, что ни в одном образце нет адаптеров выше 0.1%

---

**Вопрос:** Есть ли участки с плохим качеством (Q<20) в конце ридов?

**Ответ:** Значительного снижения качества оснований в конце ридов не наблюдается. Основная часть значений Phred score остаётся на высоком уровне (35–37), что свидетельствует о хорошем качестве секвенирования и отсутствии выраженных низкокачественных участков (Q < 20)

---

2. Написал SLURM скрипт для запуска fastp (или Trimmomatic) который:

- Обрежет адаптеры (auto-detection): `--detect_adapter_for_pe`
- Применит скользящее окно (5:20) для удаления плохих участков: `--cut_right; --cut_window_size 5; --cut_mean_quality 20`
- Отбросит риды короче 36 п.н.: `--length_required 36`
- Сохранит очищенные файлы

**run_fastp.slurm**

```bash
#!/bin/bash

#SBATCH --job-name=fastp_trim
#SBATCH --output=fastp_trim.out
#SBATCH --error=fastp_trim.err
#SBATCH --cpus-per-task=8
#SBATCH --mem=8G
#SBATCH --time=02:00:00
#SBATCH --partition=AMD9554


source ~/.bashrc
conda activate bioinf

mkdir -p ~/hw/hw5/data/trimmed

for file in ~/hw/hw5/data/raw/*.fastq.gz
do
    base=$(basename $file .fastq.gz)

    fastp \
        -i $file \
        -o ~/hw/hw5/data/trimmed/${base}_trimmed.fastq.gz \
        --detect_adapter_for_pe \
        --cut_right \
        --cut_window_size 5 \
        --cut_mean_quality 20 \
        --length_required 36 \
        -w 8 \
        -h ~/hw/hw5/data/trimmed/${base}.html \
        -j ~/hw/hw5/data/trimmed/${base}.json
done
```

---

3. Скриншот папки на сервере c созданными файлами тримминга (команда `ls -la --time=ctime`)


![image](pictures/trimmed.png)


## Часть 3 — Контроль качества после тримминга 

---

1. Запустил FastQC повторно на *_trimmed.fastq.gz и снова соберал отчет MultiQC, с помощь следующего скрипта:

**fastqc_trimmed.slurm**

```bash
#!/bin/bash

#SBATCH --job-name=fastqc_trimmed
#SBATCH --output=fastqc_trimmed.out
#SBATCH --error=fastqc_trimmed.err
#SBATCH --cpus-per-task=8
#SBATCH --mem=8G
#SBATCH --time=02:00:00
#SBATCH --partition=AMD9554

source ~/.bashrc
conda activate bioinf

mkdir -p ~/hw/hw5/results/fastqc_trimmed
mkdir -p ~/hw/hw5/results/multiqc_trimmed

fastqc ~/hw/hw5/data/trimmed/*_trimmed.fastq.gz \
       -o ~/hw/hw5/results/fastqc_trimmed \
       -t 8

multiqc ~/hw/hw5/results/fastqc_trimmed \
       -o ~/hw/hw5/results/multiqc_trimmed
```

2. Сравнил до и после. Если отвечать кратко, то метрики улучшились, но не сильно. Мысли по этому поводу излил в Выводе.

**Вывод:** Анализ качества после тримминга показал, что общее качество данных улучшилось, но незначительно

Метрики качества оснований (Per base sequence quality) и распределение нуклеотидов (Per base sequence content) практически не изменились, что скорее всего объясняется изначально высоким качеством исходных ридов

Наблюдали улучшение распределения GC-содержания (уже нет красного, только зеленое и желтое)

Наиболее заметные изменения связаны с распределением длин последовательностей: после тримминга уменьшилось количество коротких ридов, что отражает удаление последовательностей длиной менее 36 п.н.

Итого, тримминг не оказал значительного влияния на качество оснований, однако позволил улучшить структуру данных за счёт удаления коротких и потенциально менее информативных ридов